# Telecom-T2C — Benchmark Only (PASS_0-4 accuracy against an existing adapter)

Re-runs generation-based evaluation (`exact_match_rate` + per-`PASS_0-4`
accuracy) against an **already-trained** adapter on Google Drive — no
training, no dataset statistics, no fresh model/adapter load beyond what
`benchmark.run_benchmark()` needs internally. Use this to check whether a
code fix (e.g. `inference.build_prompt()`'s thinking-channel fix) actually
improved generation quality, without re-running the full multi-hour
`Telecom_T2C_Trainer_v2.ipynb` notebook.

**This notebook only orchestrates** — all logic lives in `src/`, same as
the trainer notebook. Sections: Sync Code + Mount Drive, Runtime Check,
Install, Configuration, Locate Adapter, Load Validation/Golden Dataset,
Run Benchmark.

Writes `benchmark_report.json` + `<source>_predictions.jsonl` locally, then
syncs both back into the **same** Drive run directory the adapter came
from (alongside `adapter/`), so results stay organized per-run.

## 0. Sync Code + Mount Google Drive

In [ ]:
import os
import subprocess

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/sandeep-gupta-azalio/Telecom-T2C.git"
REPO_DIR = "/content/Telecom-T2C"

if IN_COLAB:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        print(f"Repo already present at {REPO_DIR} -- pulling latest changes...")
        status = subprocess.run(
            ["git", "-C", REPO_DIR, "status", "--porcelain"], capture_output=True, text=True
        ).stdout
        stashed = bool(status.strip())
        if stashed:
            print("Local changes detected (e.g. an edited configs/experiment.yaml) -- stashing before pull.")
            subprocess.check_call(["git", "-C", REPO_DIR, "stash", "--include-untracked"])
        subprocess.check_call(["git", "-C", REPO_DIR, "pull"])
        if stashed:
            try:
                subprocess.check_call(["git", "-C", REPO_DIR, "stash", "pop"])
            except subprocess.CalledProcessError:
                print(
                    "WARNING: could not automatically restore your local changes (merge conflict "
                    "with the pulled update) -- run `!git -C /content/Telecom-T2C stash show -p` in "
                    "a new cell to recover them manually, then resolve and `git stash drop`."
                )
    else:
        print(f"Cloning {REPO_URL} into {REPO_DIR}...")
        subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])

    os.chdir(REPO_DIR)

    from google.colab import drive

    drive.mount("/content/drive")
else:
    print(
        "Not running in Colab -- skipping repo sync and Drive mount. "
        "Make sure your working directory is already inside the Telecom-T2C repo."
    )

print(f"cwd: {os.getcwd()}")


## 1. Runtime Check

In [ ]:
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Locate the Telecom-T2C project root from any Colab/local starting cwd."""
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for child in start.glob("*/"):
        if (child / "src").is_dir() and (child / "configs").is_dir():
            return child
    raise RuntimeError(
        "Could not locate the Telecom-T2C project root (a directory containing both "
        "'src/' and 'configs/'). If running in Colab, cd into the cloned/uploaded repo "
        "directory first, e.g.:\n  %cd /content/Telecom-T2C"
    )


PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src import utils

logger = utils.setup_logging()
gpu = utils.detect_gpu()
print(f"GPU: {gpu.name} | family={gpu.family} | vram={gpu.vram_gb:.1f} GB | bf16={gpu.bf16_supported}")

import torch

print(f"torch: {torch.__version__} | built for CUDA {torch.version.cuda}")
print(
    "Note: requirements.txt deliberately never pins/reinstalls torch — Colab's "
    "pre-installed build is already matched to its own CUDA driver and to "
    "torchaudio/torchvision. If a later `pip install` step ever reports a "
    "different torch version here after a restart, see README Troubleshooting "
    "('PyTorch and TorchAudio were compiled with different CUDA versions')."
)

if gpu.family == "CPU":
    raise RuntimeError(
        "No GPU detected. In Colab: Runtime -> Change runtime type -> select a GPU "
        "(A100 recommended for this 12B model)."
    )
elif gpu.family != "A100":
    print(
        f"Warning: this notebook's defaults are tuned for A100. Detected {gpu.family} — "
        "Section 7 (Load Model) will print recommended overrides for configs/experiment.yaml."
    )

## 2. Install

Same phased install as `Telecom_T2C_Trainer_v2.ipynb` Section 2 — running
generation-based evaluation needs the identical Unsloth/transformers stack
used for training (see that notebook / `requirements.txt`'s top comment for
the full reasoning).

In [ ]:
import re
import subprocess
import sys

import torch

from src import utils

# Phase 1: everything that resolves normally (no --no-deps needed) —
# huggingface_hub, datasets, sentencepiece, PyYAML, protobuf, hf_transfer,
# safetensors, wandb, nvidia-ml-py, pytest. --upgrade is required, not
# optional: without it, pip leaves an already-installed package alone as
# long as it satisfies a floor constraint (e.g. peft==0.14.0 already
# "satisfies" peft>=0.14.0), so a re-run after pulling an updated
# requirements.txt would otherwise silently do nothing.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "-r", str(PROJECT_ROOT / "requirements.txt"),
    ]
)

# Phase 2: the correlated Unsloth ML stack, installed together with
# --no-deps so pip never attempts to resolve unsloth_zoo's declared
# transformers<=5.5.0 ceiling against this project's actual transformers
# version (installed separately in Phase 3) — see requirements.txt's top
# comment for the full, empirically-confirmed reasoning. Floors on
# bitsandbytes/accelerate/peft/trl match this project's own
# previously-validated versions (--no-deps skips dependency RESOLUTION,
# not version constraints on the packages named directly in this command).
# xformers' exact pin is chosen dynamically from the installed torch
# version, copied verbatim from Unsloth's own official Colab recipe (see
# notebook Section 2 markdown for the link) since picking the wrong one
# for a given torch release is a real, separate compatibility trap.
torch_version = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
xformers_pin = "xformers==" + {
    "2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2",
}.get(torch_version, "0.0.34")
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "unsloth_zoo", "bitsandbytes>=0.46.1,!=0.48.0", "accelerate>=1.8",
        xformers_pin, "peft>=0.19.1", "trl>=0.15.0", "triton", "unsloth",
    ]
)

# Phase 3: torchao, also --no-deps and kept separate from Phase 2 (matching
# the official recipe) — unsloth_zoo declares torchao>=0.13.0 as a genuine
# dependency of its own (not just something transformers' quantizer
# machinery incidentally imports), so this project installs a real,
# specific floor for it rather than uninstalling it outright. An earlier
# version of this cell uninstalled torchao entirely after hitting a
# present-but-broken torchao/torch op mismatch
# (`torch.ops._c10d_functional._wrap_tensor_autograd`); this floor mirrors
# what Unsloth's own official notebook installs for a newer Gemma 4
# variant, on the theory that an unconstrained `pip install torchao`
# (letting pip pick whatever's latest) was the actual cause of that
# mismatch, not torchao categorically. utils.disable_unused_transformers_backends()
# below still neutralizes transformers' OWN torchao-mediated quantizer
# import path regardless (this project never uses TorchAO quantization
# directly), as a second line of defense in case this floor still doesn't
# match a given Colab image's torch build.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "torchao>=0.16.0",
    ]
)

# Phase 4: transformers + tokenizers, --no-deps, installed LAST and
# separately from Phase 2 — this is what actually gets this project past
# unsloth_zoo's declared transformers<=5.5.0 ceiling. transformers==5.10.2
# is the narrow window that both recognizes google/gemma-4-12B-it's
# `gemma4_unified` architecture (registered starting at transformers
# 5.10.0) and stays below the version where unsloth's own exec()-based
# patching (unsloth/models/_utils.py) is confirmed broken (5.12.1 raised
# `NameError: name 'auto_docstring' is not defined`) — see
# requirements.txt's top comment for the full history. tokenizers' range
# matches transformers==5.10.2's own declared requirement exactly.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "transformers==5.10.2", "tokenizers>=0.22.0,<=0.23.0",
    ]
)

# torchaudio is not needed anywhere in this text-only project, and has been
# confirmed — in a separate incident from the torchao one above — to ship
# on some Colab images with an internal circular-import bug in its own
# CUDA-version check, which crashes unrelated imports (nearly any Auto*
# class transitively pulls it in via transformers' generic audio-loss
# module). Uninstalling it entirely is the fix: transformers' own
# optional-dependency detection correctly skips torchaudio-dependent code
# paths when the package isn't present at all, rather than attempting to
# use a present-but-broken one. check=False because it's a no-op (not an
# error) if it isn't installed in the first place.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"],
    check=False,
)

# Belt-and-suspenders on top of Phase 3 above: transformers' own
# is_torchao_available() check only confirms torchao is *present*, not that
# importing it actually succeeds — a real, previously-confirmed failure
# mode on Colab images where the installed torchao build doesn't match the
# installed torch build. This project never uses TorchAO quantization
# directly (always bitsandbytes 4-bit via Unsloth), so neutralizing this
# check entirely is safe regardless of whether Phase 3's floor actually
# works on a given image. Called as early as possible (before Section 2's
# own diagnostic `import unsloth` below, and well before Sections 4/7's
# transformers imports) since the check is `@lru_cache`d — see
# utils.disable_unused_transformers_backends()'s docstring for the full
# reasoning.
utils.disable_unused_transformers_backends()

## 3. Configuration

Loads `configs/experiment.yaml` — same config the training run used, so the
adapter reloads against the exact base model it was trained against.

In [ ]:
from src import config as config_mod
from src import tokenizer as tokenizer_mod

CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment.yaml"
experiment_config = config_mod.load_config(CONFIG_PATH)
hf_token = tokenizer_mod.resolve_hf_token(experiment_config.model.hf_token_env_var)
print(f"base_model: {experiment_config.model.base_model}")


## 4. Locate Adapter on Google Drive

Auto-detects the most recently synced `run_*/adapter/` under
`drive.google_drive_directory`. Set `ADAPTER_RUN_OVERRIDE` below (a run
directory name, e.g. `"run_20260725_084649"`) to target a specific run
instead of the latest one.

In [ ]:
from pathlib import Path

from src import checkpoint as checkpoint_mod
from src import utils

ADAPTER_RUN_OVERRIDE = None  # e.g. "run_20260725_084649"

drive_base = Path(experiment_config.drive.google_drive_directory)

if ADAPTER_RUN_OVERRIDE:
    source_run_dir = drive_base / ADAPTER_RUN_OVERRIDE
else:
    source_run_dir = checkpoint_mod.find_latest_synced_run(drive_base)
    if source_run_dir is None:
        raise RuntimeError(
            f"No synced run with an adapter/ directory found under {drive_base}. "
            "Set ADAPTER_RUN_OVERRIDE above to a specific run directory name."
        )

RUN_NAME = source_run_dir.name
adapter_dir = source_run_dir / "adapter"
print(f"Benchmarking adapter: {adapter_dir}")

# Local scratch dir reusing the SAME run name, so Section 6 syncs
# predictions/report back into this run's own Drive folder (next to
# adapter/), not a brand-new run directory.
run_dir = utils.ensure_dir(PROJECT_ROOT / "outputs" / "runs" / RUN_NAME)


## 5. Load Validation/Golden Dataset

Only what `benchmark.run_benchmark()` needs for generation-based eval — no
train split, no token-length statistics.

In [ ]:
from src import dataset as dataset_mod

tokenizer_for_loading = tokenizer_mod.load_tokenizer(experiment_config.model.base_model, hf_token)
loader = dataset_mod.DatasetLoader(experiment_config.data, tokenizer_for_loading, experiment_config.identity.seed)

val_ds = loader.load_validation()
golden_ds = loader.load_golden()
print(f"val rows: {len(val_ds) if val_ds is not None else 0}")
print(f"golden rows: {len(golden_ds) if golden_ds is not None else 0}")


## 6. Run Benchmark

Reloads the adapter fresh via `inference.load_model_for_inference()` (same
path the trainer notebook's Smoke Test and the inference-server notebook
use — never merges the adapter), runs generation against golden (if
configured) or falls back to the validation set, scores `exact_match_rate`
and per-`PASS_0-4` accuracy, then syncs the report + predictions back to
this run's Drive folder.

In [ ]:
from src import benchmark as benchmark_mod
from src import checkpoint as checkpoint_mod

report = benchmark_mod.run_benchmark(
    experiment_config, adapter_dir, run_dir, golden_ds,
    hf_token=hf_token, fallback_dataset=val_ds,
)
benchmark_mod.write_benchmark_report(report, run_dir / "metrics" / "benchmark_report.json")

print(f"Generation-eval dataset used: {report.eval_dataset_source}")
print("Exact-match metrics:", report.golden_metrics)
print("Per-PASS accuracy (PASS_0-4):")
for pass_name, pass_stats in report.pass_metrics.items():
    print(f"  {pass_name}: {pass_stats}")

checkpoint_mod.sync_run_to_drive(
    run_dir, experiment_config.drive.google_drive_directory, experiment_config.drive.drive_mount_point
)
print(f"Synced predictions/report to {experiment_config.drive.google_drive_directory}/{RUN_NAME}/")
